# Ashes 3D Engine — TRELLIS Colab Worker

Free prototype worker: upload one clean product image or a 2×2 multi-view sheet, reconstruct a textured 3D asset with Microsoft TRELLIS, and download a web-ready GLB.

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**. TRELLIS requires Linux and at least 16 GB NVIDIA VRAM. A free T4 is the minimum and may occasionally run out of memory.

TRELLIS is MIT licensed. This notebook is an Ashes AI modification/integration; Microsoft does not sponsor or endorse Ashes AI.

In [ ]:
# 1. Verify the GPU before installing anything
import subprocess, re
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True).strip()
print('GPU:', gpu)
memory_mb = int(re.search(r'(\d+) MiB', gpu).group(1))
assert memory_mb >= 15000, 'TRELLIS needs approximately 16 GB VRAM. Select a T4/L4/A100 GPU runtime.'

In [ ]:
# 2. Clone Microsoft TRELLIS and install the pinned CUDA environment
# Run this cell once. It deliberately restarts the runtime at the end so Colab
# does not keep an old NumPy binary loaded after dependency installation.
!rm -rf /content/TRELLIS
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
%cd /content/TRELLIS
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!bash -lc 'cd /content/TRELLIS && source setup.sh --basic --xformers --diffoctreerast --spconv --mipgaussian --kaolin --nvdiffrast'
print('Installation complete. Restarting the Colab runtime now...')
import os, signal
os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
# 3. Upload one product image OR one 2×2 multi-view sheet
from google.colab import files
from PIL import Image
from pathlib import Path
import io, os

SPLIT_2X2_SHEET = True  # False for a normal single product photograph
uploaded = files.upload()
assert uploaded, 'Upload a PNG or JPG.'
name, raw = next(iter(uploaded.items()))
source = Image.open(io.BytesIO(raw)).convert('RGBA')
work = Path('/content/ashes_input'); work.mkdir(exist_ok=True)

if SPLIT_2X2_SHEET:
    w, h = source.size
    boxes=[(0,0,w//2,h//2),(w//2,0,w,h//2),(0,h//2,w//2,h),(w//2,h//2,w,h)]
    images=[source.crop(box) for box in boxes]
else:
    images=[source]

for i,image in enumerate(images):
    image.thumbnail((1024,1024), Image.Resampling.LANCZOS)
    image.save(work/f'view_{i+1}.png')
display(*images)
print(f'Prepared {len(images)} consistent view(s).')

In [ ]:
# 4. Isolate the product without rembg (avoids its Python 3.12/NumPy conflict)
# This estimates the studio background from the four corners and fades pixels
# close to that colour. If an uploaded PNG already has transparency, it keeps it.
from PIL import Image

def border_background_cutout(image, threshold=34, feather=30):
    rgba=image.convert('RGBA')
    w,h=rgba.size
    px=rgba.load()
    sample=max(1,min(w,h)//18)
    corners=[]
    for x0,y0 in [(0,0),(w-sample,0),(0,h-sample),(w-sample,h-sample)]:
        for y in range(y0,min(h,y0+sample)):
            for x in range(x0,min(w,x0+sample)):
                corners.append(px[x,y][:3])
    bg=tuple(sum(c[i] for c in corners)//len(corners) for i in range(3))
    out=rgba.copy(); op=out.load()
    for y in range(h):
        for x in range(w):
            r,g,b,a=op[x,y]
            distance=((r-bg[0])**2+(g-bg[1])**2+(b-bg[2])**2)**0.5
            inferred=max(0,min(255,int(255*(distance-threshold)/feather)))
            op[x,y]=(r,g,b,min(a,inferred))
    return out

prepared=[]
for i,image in enumerate(images):
    has_alpha=image.mode=='RGBA' and image.getextrema()[3][0] < 255
    cutout=image.convert('RGBA') if has_alpha else border_background_cutout(image)
    bbox=cutout.getbbox()
    if bbox: cutout=cutout.crop(bbox)
    canvas=Image.new('RGBA',(768,768),(255,255,255,0))
    cutout.thumbnail((680,680),Image.Resampling.LANCZOS)
    canvas.alpha_composite(cutout,((768-cutout.width)//2,(768-cutout.height)//2))
    canvas.save(work/f'prepared_{i+1}.png')
    prepared.append(canvas)
display(*prepared)


In [ ]:
# 5. Load TRELLIS and reconstruct hidden geometry
import os, sys, torch
os.environ['ATTN_BACKEND']='xformers'
os.environ['SPCONV_ALGO']='native'
sys.path.insert(0,'/content/TRELLIS')
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils

torch.cuda.empty_cache()
pipeline=TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipeline.cuda()
params=dict(
    seed=42,
    sparse_structure_sampler_params={'steps':12,'cfg_strength':7.5},
    slat_sampler_params={'steps':12,'cfg_strength':3.0},
)
if len(prepared)>1:
    outputs=pipeline.run_multi_image(prepared,**params)
else:
    outputs=pipeline.run(prepared[0],**params)
print('Reconstruction finished.')

In [ ]:
# 6. Export an Ashes web-ready textured GLB and turntable preview
import imageio
out_dir=Path('/content/ashes_output'); out_dir.mkdir(exist_ok=True)
glb=postprocessing_utils.to_glb(
    outputs['gaussian'][0],
    outputs['mesh'][0],
    simplify=0.95,
    texture_size=1024,
)
glb_path=out_dir/'ashes-product.glb'
glb.export(glb_path)
frames=render_utils.render_video(outputs['gaussian'][0])['color']
preview_path=out_dir/'ashes-turntable.mp4'
imageio.mimsave(preview_path,frames,fps=30)
print('GLB:',glb_path,glb_path.stat().st_size//1024,'KB')
print('Preview:',preview_path)

In [ ]:
# 7. Preview and download
from IPython.display import Video, display
display(Video(str(preview_path),embed=True,width=640))
files.download(str(glb_path))
files.download(str(preview_path))